# 📊 Real Estate Price Prediction

> **Scenario:** A new real estate company lets homeowners sell online at a small discount from estimated value, then resells at a higher price. 
The model must **accurately predict selling price** — overpay = loss, underpay = no deal.  
> **Competitive Advantage:** Proprietary database of public + private housing data and historical transactions.  
> **Objective:** Build an ML model to set the buy price the company offers homeowners.

---

### Table of Contents
1. Define the Problem  
2. Data Collection & Cleaning  
3. Exploratory Data Analysis  
4. Feature Engineering  
5. Data Splitting  
6. Baseline Model  
7. Hyperparameter Tuning  
8. Model Evaluation  
9. Iteration  
10. Interpretation & Communication

---
## 1 · Define the Problem

- **Goal:** Predict house selling price → company offers `predicted_price × (1 − discount_rate)` to homeowners  
- **Primary KPI:** RMSE — large errors = direct financial loss  
- **Secondary KPIs:** MAE, R², % of predictions within ±10 % of actual  
- **Over-prediction risk:** Company pays too much, resells at a loss  
- **Under-prediction risk:** Homeowners reject the offer (lost volume)  
- **Why ML:** Captures non-linear relationships that traditional comps-based appraisals miss

---
## 2 · Data Collection & Cleaning

Load the proprietary dataset (public + private housing data), then clean:
- Handle missing values (imputation)  
- Rename columns for clarity (snake_case)  
- Correct data types  
- Detect and address outliers (IQR)

#### 2.1 · Setup

In [1]:
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# -----------------------------
# Add project root to sys.path for module imports
# -----------------------------
root = "/Users/phillipsmith/Desktop/pythonProjects/real-estate-price-prediction"
sys.path.append(root)

# -----------------------------
# Import external python modules
# -----------------------------
from pathlib import Path
from src.preprocessing import data_prep

# -----------------------------
# Display full dataframes
# -----------------------------
pd.set_option('display.max_rows', None) # reset_option to compact dataframe view
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.6f}'.format)

# -----------------------------
# Define path to raw Excel data source
# -----------------------------
excel_path = Path(
    '/Users/phillipsmith/Desktop/pythonProjects/real-estate-price-prediction/src/data/raw/Dataset Project 2.xlsx'
)

# -----------------------------
# Read raw data from Excel source
# -----------------------------
df_raw = pd.read_excel(excel_path, skiprows=1)
print(f"Raw DataFrame shape: {df_raw.shape}")

df_raw.head()

Raw DataFrame shape: (506, 15)


,Unnamed: 0,crim,zn,indus,chas,nox,rm,age,dis,rad,tax,ptratio,popul,lstat,price
0,1,0.006320,18.000000,2.310000,0,0.538000,6.575000,65.200000,4.090000,1,296,15.300000,396.900000,4.980000,24.000000
1,2,0.027310,0.000000,7.070000,0,0.469000,6.421000,78.900000,4.967100,2,242,17.800000,396.900000,9.140000,21.600000
2,3,0.027290,0.000000,7.070000,0,0.469000,7.185000,61.100000,4.967100,2,242,17.800000,392.830000,4.030000,34.700000
3,4,0.032370,0.000000,2.180000,0,0.458000,6.998000,45.800000,6.062200,3,222,18.700000,394.630000,2.940000,33.400000
4,5,0.069050,0.000000,2.180000,0,0.458000,7.147000,54.200000,6.062200,3,222,18.700000,396.900000,5.330000,36.200000


#### 2.2 · Data Cleaning

In [2]:
df_cleaned = data_prep.clean_data(df_raw)

# statistical summary
df_cleaned.describe()

,crime_rate_per_capita_crim,large_lot_zoning_ratio_zn,non_retail_acre_ratio_indus,river_boundary_flag_chas,nox_concentration_nox,avg_rooms_per_dwelling_rm,pre_1940_housing_ratio_age,employment_center_distance_dis,radial_highway_access_idx_rad,property_tax_rate_tax,pupil_teacher_ratio_ptratio,population_distribution_popul,low_ses_population_pct_lstat,price
count,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000
mean,3.613524,11.363636,11.136779,0.069170,0.554695,6.284634,68.574901,3.795043,9.549407,408.237154,18.455534,356.674032,12.653063,22.532806
std,8.601545,23.322453,6.860353,0.253994,0.115878,0.702617,28.148861,2.105710,8.707259,168.537116,2.164946,91.294864,7.141062,9.197104
min,0.006320,0.000000,0.460000,0.000000,0.385000,3.561000,2.900000,1.129600,1.000000,187.000000,12.600000,0.320000,1.730000,5.000000
25%,0.082045,0.000000,5.190000,0.000000,0.449000,5.885500,45.025000,2.100175,4.000000,279.000000,17.400000,375.377500,6.950000,17.025000
50%,0.256510,0.000000,9.690000,0.000000,0.538000,6.208500,77.500000,3.207450,5.000000,330.000000,19.050000,391.440000,11.360000,21.200000
75%,3.677083,12.500000,18.100000,0.000000,0.624000,6.623500,94.075000,5.188425,24.000000,666.000000,20.200000,396.225000,16.955000,25.000000
max,88.976200,100.000000,27.740000,1.000000,0.871000,8.780000,100.000000,12.126500,24.000000,711.000000,22.000000,396.900000,37.970000,50.000000


#### 2.3 · Missing Values
Check for nulls. Apply imputation as needed (mean/median for numeric, mode for categorical).

In [3]:
print(df_cleaned[df_cleaned.columns].isna().sum()) # None

crime_rate_per_capita_crim        0
large_lot_zoning_ratio_zn         0
non_retail_acre_ratio_indus       0
river_boundary_flag_chas          0
nox_concentration_nox             0
avg_rooms_per_dwelling_rm         0
pre_1940_housing_ratio_age        0
employment_center_distance_dis    0
radial_highway_access_idx_rad     0
property_tax_rate_tax             0
pupil_teacher_ratio_ptratio       0
population_distribution_popul     0
low_ses_population_pct_lstat      0
price                             0
dtype: int64


#### 2.4 · Data Transformations

In [9]:
df_transformed = df_cleaned.copy()

# percentage columns to decimals
percentage_cols = ['large_lot_zoning_ratio_zn', 'non_retail_acre_ratio_indus', 'pre_1940_housing_ratio_age', 'low_ses_population_pct_lstat']
df_transformed[percentage_cols] = df_transformed[percentage_cols] / 100

# tax rate to decimals
df_transformed['property_tax_rate_tax'] = df_transformed['property_tax_rate_tax'] / 10000

# price unit to dollars
df_transformed['price'] = df_transformed['price'] * 1000

df_transformed.head()

,crime_rate_per_capita_crim,large_lot_zoning_ratio_zn,non_retail_acre_ratio_indus,river_boundary_flag_chas,nox_concentration_nox,avg_rooms_per_dwelling_rm,pre_1940_housing_ratio_age,employment_center_distance_dis,radial_highway_access_idx_rad,property_tax_rate_tax,pupil_teacher_ratio_ptratio,population_distribution_popul,low_ses_population_pct_lstat,price
0,0.006320,0.180000,0.023100,0,0.538000,6.575000,0.652000,4.090000,1,0.029600,15.300000,396.900000,0.049800,24000.000000
1,0.027310,0.000000,0.070700,0,0.469000,6.421000,0.789000,4.967100,2,0.024200,17.800000,396.900000,0.091400,21600.000000
2,0.027290,0.000000,0.070700,0,0.469000,7.185000,0.611000,4.967100,2,0.024200,17.800000,392.830000,0.040300,34700.000000
3,0.032370,0.000000,0.021800,0,0.458000,6.998000,0.458000,6.062200,3,0.022200,18.700000,394.630000,0.029400,33400.000000
4,0.069050,0.000000,0.021800,0,0.458000,7.147000,0.542000,6.062200,3,0.022200,18.700000,396.900000,0.053300,36200.000000


#### 2.5 · Outlier Detection

---
## 3 · Exploratory Data Analysis (EDA)

#### 3.1 · Plot Distributions
- **Histograms / density (KDE) plots** for every numeric feature  
- Look for skewness → candidates for log or Box-Cox transforms  
- Check target variable (`price`) distribution — heavy right skew is common in real estate

#### 3.2 · Correlations
- **Pearson heatmap** for linear relationships  
- **Pairplots** for bivariate scatter patterns  
- Watch for multicollinearity (|r| > 0.8 between features)

#### 3.3 · Feature Importance (Quick Screen)
- Fit a simple `RandomForestRegressor` or use `mutual_info_regression` from sklearn  
- Rank features by importance to guide feature selection

In [5]:
"""Seaborn and Line Plots"""
# plt.figure(figsize=(16,6))
# sns.lineplot(data=df['monthly_revenue']) # original

"""Line Plots"""
# plt.figure(figsize=(20,6))
# plt.title(current_var)
# sns.lineplot(data=df[current_var])

"""Bar Plots"""
# x_var = 'revenue_bin'
# y_var = 'rated_exposure'
# plt.figure(figsize=(25,8))
# plt.title(f'{x_var} vs. {y_var}')
# sns.barplot(x=monthly_revenue_raw[x_var], y=df_transformed[y_var])

"""Scatter Plots"""
# x_var = 'rated_exposure'
# y_var = 'age'
# plt.figure(figsize=(25,7))
# plt.title(f'{x_var} vs. {y_var}')
# sns.scatterplot(x=df[x_var], y=df[y_var])

"""Pie Graphs (Custom)"""
# plt.figure(figsize=(25,9))
# Plots.plot_age_ranges(df, 'age', title='% of Age Ranges')

"""Correlation Matrix"""
# correlation_df = df_transformed[[
#     ]].copy()

# correlation_matrix = correlation_df.corr()
# plt.figure(figsize=(15,9))
# sns.heatmap(correlation_matrix, annot=True, cmap='YlGnBu')

plt.show()

---
## 4 · Feature Engineering

#### 4.1 · Feature Selection & DataFrame Modeling
Select independent variables (X) and dependent variable (y = `price`). Display summary statistics for the final feature set.

#### 4.2 · Techniques
| Technique | Example |
|---|---|
| **Interaction terms** | `lot_area × overall_qual` |
| **Log transform** | `np.log1p(price)` to reduce right skew |
| **Box-Cox transform** | `scipy.stats.boxcox` for optimal power transform |
| **Standardization** | `(x − μ) / σ` — required for Lasso / Ridge |
| **Min-Max normalization** | Scale to [0, 1] — useful for distance-based models |
| **Domain features** | `age_of_home = year_sold − year_built`, `price_per_sqft` |

> 🔑 Domain knowledge often produces the most impactful features.

In [6]:
modeling_df = df_transformed[[

]]

---
## 5 · Data Splitting

#### 5.1 · Train/Test Split
Split data into training and testing sets. Start with 80/20, then experiment with 50/50 and 95/5 splits.

#### 5.2 · Strategy
- **Stratified sampling** — bin `price` into quantiles and stratify to ensure representative splits  
- **Time-based splits** — if temporal data exists (e.g., `year_sold`), use a cutoff date  
- **Multiple split ratios** — 80/20 (default), 50/50 (stress test), 95/5 (data-hungry models)

---
## 6 · Baseline Model

#### 6.1 · Linear Regression
Train OLS linear regression. Evaluate with RMSE, MAE, R².

#### 6.2 · Naïve Baseline & Cross-Validation
- Naïve baseline: predict mean/median price. Record metrics.  
- k-fold cross-validation (k = 5 or 10) to assess stability.  
- All subsequent models must beat the naïve baseline.

In [7]:
"""Initialize Variables"""
modeling_df = modeling_df.copy()

---
## 7 · Hyperparameter Tuning

#### 7.1 · Model 2 — Random Forest
Train a Random Forest regressor. Compare performance against linear regression.

#### 7.2 · Model 3 — Lasso & Ridge Regression
Train Lasso and Ridge variants. Evaluate how regularization affects prediction accuracy.

#### 7.3 · Scaling & Normalization
Apply `StandardScaler` and/or `MinMaxScaler` to features. Re-train models and compare results.

#### 7.4 · Tuning Techniques
- **Grid Search** (`GridSearchCV`) — exhaustive, best for small param spaces  
- **Random Search** (`RandomizedSearchCV`) — faster, good default  
- **Bayesian Optimization** (`optuna`) — efficient for large/continuous spaces

---
## 8 · Model Evaluation

| Metric | Business Meaning |
|---|---|
| **RMSE** | Penalises large mispricings — direct financial loss per deal |
| **MAE** | Average dollar error on buy-price offers |
| **R²** | Proportion of price variance explained |
| **MAPE** | Must be < discount rate for the company to profit |

#### 8.1 · Model Comparison & Back-Testing
Compare all models across 80/20, 50/50, and 95/5 splits. Select the best performer.

Plot residuals vs. predicted. Check for overfitting (high train R² + low test R²).

---
## 9 · Iteration

1. Analyse worst predictions — look for common patterns  
2. Refine features based on importance and error analysis  
3. Experiment with tree-based / ensemble models (XGBoost, LightGBM, stacking)  
4. Repeat tuning and evaluation

---
## 10 · Interpretation & Communication

#### 10.1 · Results & Conclusions
Summarize best model, impact of scaling, and effect of split ratios.

#### 10.2 · Business KPIs
Map RMSE/MAE back to profit-per-transaction. Confirm MAPE < discount rate.

#### 10.3 · Visualizations
- Predicted vs. Actual scatter (ideal line y = x)  
- Residual distribution  
- Feature importance (top 10 price drivers)

#### 10.4 · Limitations
- Model trained on historical data — market shifts may degrade accuracy  
- Features not captured (school ratings, market sentiment, renovation quality)